In [6]:
# ==========================================================
# DIARY EMOTION AI MODEL - COMPLETE CORRECTED CODE
# ==========================================================


# ============================
# 1. Install Libraries
# ============================

!pip install -U datasets transformers accelerate scikit-learn



# ============================
# 2. Import Libraries
# ============================

import torch
import numpy as np

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)


print("Libraries imported")



# ============================
# 3. Download Dataset
# ============================

dataset = load_dataset(
    "google-research-datasets/go_emotions",
    "simplified"
)


print(dataset)



# ============================
# 4. Label Mapping
# ============================

labels = dataset["train"].features["labels"].feature.names


id2label = {}

label2id = {}


for i, label in enumerate(labels):

    id2label[i] = label

    label2id[label] = i


print("Number of emotions:", len(labels))



# ============================
# 5. Load Tokenizer
# ============================

model_name = "distilbert-base-uncased"


tokenizer = AutoTokenizer.from_pretrained(
    model_name
)


print("Tokenizer loaded")



# ============================
# 6. Tokenize Dataset
# ============================

def tokenize_function(example):

    return tokenizer(

        example["text"],

        padding="max_length",

        truncation=True,

        max_length=128

    )



tokenized_dataset = dataset.map(

    tokenize_function,

    batched=True

)


print("Tokenization completed")



# ============================
# 7. Convert Labels
# ============================

def change_label(example):

    example["label"] = example["labels"][0]

    return example



tokenized_dataset = tokenized_dataset.map(
    change_label
)



# Remove unused columns

tokenized_dataset = tokenized_dataset.remove_columns(

    [
        "text",
        "labels"
    ]

)



# Convert dataset to torch

tokenized_dataset.set_format(
    "torch"
)


print(tokenized_dataset)



# ============================
# 8. Load DistilBERT Model
# ============================

model = AutoModelForSequenceClassification.from_pretrained(

    model_name,

    num_labels=len(labels),

    id2label=id2label,

    label2id=label2id

)


print("Model loaded successfully")



# ============================
# 9. Evaluation Function
# ============================

def compute_metrics(eval_pred):

    logits, true_labels = eval_pred


    predictions = np.argmax(
        logits,
        axis=-1
    )


    precision, recall, f1, _ = precision_recall_fscore_support(

        true_labels,

        predictions,

        average="weighted",

        zero_division=0

    )


    accuracy = accuracy_score(

        true_labels,

        predictions

    )


    return {

        "accuracy": accuracy,

        "precision": precision,

        "recall": recall,

        "f1": f1

    }



# ============================
# 10. Training Settings
# ============================

training_args = TrainingArguments(

    output_dir="./models/checkpoints",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    logging_steps=100,

    report_to="none"

)


print("Training settings created")



# ============================
# 11. Trainer
# ============================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_dataset["train"],

    eval_dataset=tokenized_dataset["validation"],

    compute_metrics=compute_metrics

)


print("Trainer created")



# ============================
# 12. Train Model
# ============================

trainer.train()



# ============================
# 13. Test Accuracy
# ============================

results = trainer.evaluate(

    tokenized_dataset["test"]

)


print(results)



# ============================
# 14. Move Model to Device
# ============================

device = torch.device(

    "cuda" if torch.cuda.is_available()

    else "cpu"

)


model.to(device)


print("Using device:", device)



# ============================
# 15. Prediction Function
# ============================

def predict_emotion(text):


    inputs = tokenizer(

        text,

        return_tensors="pt",

        truncation=True,

        padding=True,

        max_length=128

    )


    # Move input to same device as model

    inputs = {

        key:value.to(device)

        for key,value in inputs.items()

    }


    model.eval()


    with torch.no_grad():

        output = model(**inputs)



    prediction = torch.argmax(

        output.logits,

        dim=1

    )


    emotion = id2label[

        prediction.item()

    ]


    confidence = torch.softmax(

        output.logits,

        dim=1

    )


    confidence_score = torch.max(

        confidence

    ).item()



    return {

        "emotion": emotion,

        "confidence": round(confidence_score*100,2)

    }



# ============================
# 16. Test Diary
# ============================


result = predict_emotion(

    "Today I completed my project and I feel very proud"

)


print(result)



# ============================
# 17. Save Model
# ============================


model.save_pretrained(

    "models/diary_emotion_model"

)


tokenizer.save_pretrained(

    "models/diary_emotion_model"

)


print("MODEL SAVED SUCCESSFULLY")

Libraries imported
DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})
Number of emotions: 28
Tokenizer loaded
Tokenization completed
DatasetDict({
    train: Dataset({
        features: ['id', 'input_ids', 'token_type_ids', 'attention_mask', 'label'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['id', 'input_ids', 'token_type_ids', 'attention_mask', 'label'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['id', 'input_ids', 'token_type_ids', 'attention_mask', 'label'],
        num_rows: 5427
    })
})


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully
Training settings created
Trainer created


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.433368,1.402862,0.578511,0.569043,0.578511,0.555281
2,1.268792,1.374565,0.584224,0.571680,0.584224,0.571575
3,1.087642,1.409071,0.588463,0.575474,0.588463,0.576821


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (2 outputs left out)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
1.087642,1.439612,3,0.576377,0.563036,0.576377,0.565191


{'eval_loss': 1.4396123886108398, 'eval_accuracy': 0.5763773723972729, 'eval_precision': 0.5630356352573098, 'eval_recall': 0.5763773723972729, 'eval_f1': 0.565191363154615}
Using device: cuda
{'emotion': 'admiration', 'confidence': 37.41}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

MODEL SAVED SUCCESSFULLY
